# FloodLens AI — Data Story Notebook

**Mapping Chennai's flood history against its water bodies.**

This notebook is the reproducible companion to the FloodLens AI app. It:

1. Fetches **real** Chennai water bodies from **OpenStreetMap** (Overpass API, ODbL),
2. Validates and converts them to GeoJSON (the same logic the app's pipeline uses),
3. Loads the clearly-labelled **demonstration** flood-hotspot layer,
4. Computes overview stats and a **proximity analysis** (Haversine) between a flood hotspot and nearby water bodies,
5. Renders everything on an interactive **Folium / Leaflet** map inline.

> FloodLens AI is a **historical & geographical** tool. It does **not** predict floods. Proximity is a geographic fact, not a claim of causation.

**Data integrity:** every statistic below is computed from the loaded data — nothing is hardcoded. The water-body layer is real (OSM/ODbL); the flood-hotspot layer is demonstration data with real neighbourhood names and illustrative attributes.

## 0. Setup

In [ ]:
# Install dependencies (skip if already present)
# !pip install requests pandas folium
import json, math, urllib.request, urllib.parse
import pandas as pd
import folium
from folium.plugins import MarkerCluster

CHENNAI_CENTER = (13.047, 80.237)  # lat, lon
CHENNAI_BBOX = {"min_lon": 79.90, "max_lon": 80.45, "min_lat": 12.75, "max_lat": 13.35}

## 1. Fetch real water bodies from OpenStreetMap (Overpass API)

This is the **verified, real** dataset — named lakes, ponds, reservoirs and temple tanks inside the Chennai administrative area, licensed under the Open Database License (ODbL).

In [ ]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
OVERPASS_QUERY = '''
[out:json][timeout:60];
area["name"="Chennai"]["boundary"="administrative"]->.chn;
(
  way["natural"="water"](area.chn);
  relation["natural"="water"](area.chn);
  way["water"="lake"](area.chn);
  way["water"="pond"](area.chn);
  way["water"="reservoir"](area.chn);
);
out center tags 120;
'''

def fetch_water_bodies():
    data = b"data=" + urllib.parse.quote(OVERPASS_QUERY).encode()
    req = urllib.request.Request(OVERPASS_URL, data=data, headers={"User-Agent": "FloodLens/1.0"})
    with urllib.request.urlopen(req, timeout=75) as resp:
        payload = json.loads(resp.read().decode("utf-8"))
    rows = []
    for el in payload.get("elements", []):
        tags = el.get("tags", {})
        name = (tags.get("name") or "").strip()
        if not name:
            continue
        c = el.get("center") or {}
        if c.get("lon") is None or c.get("lat") is None:
            continue
        rows.append({
            "name": name,
            "lon": c["lon"], "lat": c["lat"],
            "waterbody_type": tags.get("water") or tags.get("natural") or "water",
            "osm_id": f"{el.get('type')}/{el.get('id')}",
        })
    return rows

try:
    water = fetch_water_bodies()
    print(f"Fetched {len(water)} named water bodies live from OpenStreetMap.")
except Exception as e:
    print(f"Live fetch failed ({e}); using a small bundled real-OSM snapshot.")
    water = [
        {"name": "Porur Lake", "lon": 80.1499163, "lat": 13.0311084, "waterbody_type": "lake", "osm_id": "way/23633592"},
        {"name": "Ambattur Lake", "lon": 80.1423361, "lat": 13.1063301, "waterbody_type": "reservoir", "osm_id": None},
        {"name": "Chembarambakkam Lake", "lon": 80.0577, "lat": 13.0106, "waterbody_type": "reservoir", "osm_id": None},
        {"name": "Retteri Lake", "lon": 80.2061, "lat": 13.1356, "waterbody_type": "lake", "osm_id": None},
        {"name": "Velachery Lake", "lon": 80.2176, "lat": 12.9856, "waterbody_type": "lake", "osm_id": None},
        {"name": "Pallikaranai Marsh", "lon": 80.2100, "lat": 12.9400, "waterbody_type": "wetland", "osm_id": None},
    ]

wdf = pd.DataFrame(water)
wdf.head()

## 2. Validate & clean (same rules as the app's pipeline)

Validate coordinates, keep only features inside the Chennai bounding box, and drop duplicates.

In [ ]:
def in_chennai(lon, lat, b=CHENNAI_BBOX):
    return (-180 <= lon <= 180 and -90 <= lat <= 90
            and b["min_lon"] <= lon <= b["max_lon"] and b["min_lat"] <= lat <= b["max_lat"])

before = len(wdf)
wdf = wdf[wdf.apply(lambda r: in_chennai(r["lon"], r["lat"]), axis=1)].copy()
wdf = wdf.drop_duplicates(subset=["name", "lon", "lat"]).reset_index(drop=True)
print(f"Validated: kept {len(wdf)} of {before} (rejected out-of-bbox / duplicates).")
print("Water body types:")
print(wdf["waterbody_type"].value_counts())

## 3. Demonstration flood-hotspot layer (clearly labelled)

Real Chennai neighbourhoods documented as severely affected in the 2015 floods. Depth/year values are **illustrative demonstration data**, flagged `demo=True`.

In [ ]:
hotspots = [
    ("Velachery", 80.2197, 12.9792, 2015, 2.0),
    ("T. Nagar", 80.2340, 13.0418, 2015, 0.9),
    ("Mudichur", 80.0640, 12.9080, 2015, 3.0),
    ("Adyar", 80.2574, 13.0067, 2015, 1.5),
    ("Kotturpuram", 80.2456, 13.0180, 2015, 2.2),
    ("Saidapet", 80.2235, 13.0211, 2015, 1.8),
    ("Pallikaranai", 80.2100, 12.9400, 2015, 2.5),
    ("Perungudi", 80.2420, 12.9630, 2015, 1.4),
    ("Madipakkam", 80.1990, 12.9630, 2015, 1.6),
    ("Mambalam", 80.2200, 13.0380, 2015, 1.0),
]
hdf = pd.DataFrame(hotspots, columns=["name", "lon", "lat", "event_year", "flood_depth_m"])
hdf["demo"] = True
hdf

## 4. Overview statistics (computed, never hardcoded)

In [ ]:
overview = {
    "water_bodies": len(wdf),
    "flood_hotspots": len(hdf),
    "neighbourhoods": hdf["name"].nunique(),
    "avg_demo_depth_m": round(hdf["flood_depth_m"].mean(), 2),
}
overview

## 5. Proximity analysis (Haversine great-circle distance)

For a chosen flood hotspot, find the nearest water bodies. **Proximity is geographic — it does not establish causation.**

In [ ]:
def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(min(1, math.sqrt(a)))

def nearest_water(hotspot_name, k=5):
    h = hdf[hdf["name"] == hotspot_name].iloc[0]
    d = wdf.copy()
    d["distance_km"] = d.apply(lambda r: haversine_km(h["lon"], h["lat"], r["lon"], r["lat"]), axis=1)
    return h, d.nsmallest(k, "distance_km")[["name", "waterbody_type", "distance_km"]]

hotspot, nearby = nearest_water("Velachery", k=5)
print(f"Nearest water bodies to {hotspot['name']} ({hotspot['lat']:.4f}, {hotspot['lon']:.4f}):\n")
nearby.assign(distance=nearby["distance_km"].map(lambda x: f"{x*1000:.0f} m" if x < 1 else f"{x:.2f} km"))

## 6. Interactive map (Folium / Leaflet)

Water bodies (cyan, clustered), flood hotspots (pink), and the proximity lines from the selected hotspot to its nearest water bodies.

In [ ]:
m = folium.Map(location=CHENNAI_CENTER, zoom_start=11, tiles="OpenStreetMap")

# Water bodies (clustered)
cluster = MarkerCluster(name="Water bodies (OSM, ODbL)").add_to(m)
for _, r in wdf.iterrows():
    folium.CircleMarker(
        location=(r["lat"], r["lon"]), radius=5, color="#0891b2", fill=True, fill_color="#38bdf8", fill_opacity=0.9,
        popup=f"{r['name']} ({r['waterbody_type']})<br>OSM: {r['osm_id']}",
    ).add_to(cluster)

# Flood hotspots (demo)
fg = folium.FeatureGroup(name="Flood hotspots (demo)").add_to(m)
for _, r in hdf.iterrows():
    folium.CircleMarker(
        location=(r["lat"], r["lon"]), radius=7, color="#be185d", fill=True, fill_color="#f472b6", fill_opacity=0.9,
        popup=f"{r['name']} — {r['event_year']}, depth {r['flood_depth_m']} m (DEMO)",
    ).add_to(fg)

# Proximity lines for the selected hotspot
prox = folium.FeatureGroup(name=f"Proximity: {hotspot['name']}").add_to(m)
for _, r in nearby.iterrows():
    wb = wdf[wdf["name"] == r["name"]].iloc[0]
    folium.PolyLine([(hotspot["lat"], hotspot["lon"]), (wb["lat"], wb["lon"])],
                    color="#22d3ee", weight=2, dash_array="5,5",
                    tooltip=f"{r['name']}: {r['distance_km']:.2f} km").add_to(prox)

folium.LayerControl().add_to(m)
m

## 7. Export GeoJSON (what the app consumes)

The app reads WGS84 (EPSG:4326) GeoJSON. Here's how the validated water bodies become that GeoJSON FeatureCollection.

In [ ]:
def to_geojson(df, prop_cols):
    feats = []
    for _, r in df.iterrows():
        feats.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [float(r["lon"]), float(r["lat"])]},
            "properties": {c: (None if pd.isna(r[c]) else r[c]) for c in prop_cols},
        })
    return {"type": "FeatureCollection",
            "crs": {"type": "name", "properties": {"name": "urn:ogc:def:crs:OGC:1.3:CRS84"}},
            "features": feats}

gj = to_geojson(wdf, ["name", "waterbody_type", "osm_id"])
print(f"GeoJSON FeatureCollection with {len(gj['features'])} features (WGS84).")
print(json.dumps(gj["features"][0], indent=2))
# To save: open('water_bodies.geojson','w').write(json.dumps(gj))

## Summary

This notebook reproduces the FloodLens AI data pipeline end to end: a **real** OSM water-body dataset, validated and converted to GeoJSON, a clearly-labelled demo flood layer, computed overview stats, a Haversine proximity analysis, and an interactive map — the same building blocks the live React + AWS app uses. Swap the Overpass bounding box and the pattern works for any city with open water data.

*FloodLens AI is a historical & geographical tool and does not provide flood forecasts. Water-body data © OpenStreetMap contributors (ODbL).*